## Try RAG with LLM on your custome database 

To use this notebook, run ```jupyter notebook``` from the notebook directory.
> ⚠️ **Warning:** You should run this command after setting up a development environment (-dev) as detailed  in the README file.

Install the necessary modules:

In [1]:
import os
from rag.retrieval import Retriever
from rag.utils import pretty_print, get_leaf_classes
import torch
from transformers import pipeline
from rag.models.embedding_models.embedding_models import EmbeddingModel

Available LLMs:
* TinyLlama/TinyLlama-1.1B-Chat-v1.0
* h2oai/h2o-danube3-500m-chat
* meta-llama/Llama-3.2-1B-Instruct
* google/gemma-1.1-2b-it
* google/gemma-3-1b-it
* Qwen/Qwen2.5-0.5B-Instruct
* HuggingFaceTB/SmolLM-135M-Instruct
* HuggingFaceTB/SmolLM-360M-Instruct
* HuggingFaceTB/SmolLM-1.7B-Instruct
* onnx-community/DeepSeek-R1-Distill-Qwen-1.5B-ONNX

Set your inference parameters:

In [2]:
generic_prompt = "You are an assistant, short answer using the following information: "  # You can try your own prompt
model_name = "h2oai/h2o-danube3-500m-chat"  # Must be a valid HuggingFace model path (access token might be required)
top_k = 3
reranking = True
best_k = 1
verbose = True

# To test the rag database of the repository (needs to be created)
rag_db_path = os.path.join(os.path.dirname(os.getcwd()), "src", "data", "rag_database.pkl")

To try LLM with RAG, run:

In [3]:
# define the LLM
pipe = pipeline(
    "text-generation",
    model=model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"\rLLM model used: {model_name}")

# RAG's retriever
retriever = Retriever(top_k=top_k,
                      reranking=reranking,
                      best_k=best_k,
                      rag_db_path=rag_db_path,
                      verbose=verbose)

while True:
    # Ask the user for a question
    user_input = input("Ask a question (or type 'q' to quit): ")

    # Check if the user pressed only Enter (empty input)
    if user_input == '':
        continue  # Do nothing and continue the loop

    # Check if the user wants to quit
    if user_input == 'q':
        print("Exiting the program.")
        break

    # contextual information is retrieved based on the user query
    chunk_list, similarity_list, metadata_list = retriever(query=user_input)

    # Command detection
    if "intent" in metadata_list[0]:
        pretty_print(name="Intent detected", result_dictionary={
        "Command n°": metadata_list[0]['intent'],
        "LLM is bypassed": True
        })
        continue

    rag_prompt = generic_prompt
    if len(chunk_list) > 0:
        rag_prompt += ' '.join(chunk_list)
    else:
        rag_prompt += "No information."
    rag_prompt += '\n' + user_input
    
    messages = [{"role": "user", "content": rag_prompt}]

    prompt = pipe.tokenizer.apply_chat_template(messages,
                                                tokenize=False,
                                                add_generation_prompt=True)
    
    res = pipe(prompt,
               return_full_text=False,
               max_new_tokens=256)
    
    llm_output = res[0]["generated_text"]

    # Print the retrieved results
    pretty_print(name="LLM", result_dictionary={
        "Prompt": prompt,
        "Answer": llm_output,
    })

LLM model used: h2oai/h2o-danube3-500m-chat
Embedding model used: all-MiniLM-L6-v2.onnx 
  ====================== RAG database information ===================== 
 - Description:   A health guide for people with diabetes.
 - Embedding model used for generation:   all-MiniLM-L6-v2.onnx
 - Chunk files used for generation:  
   ['Medical_HiRAG_chunks.json']



Ask a question (or type 'q' to quit):  What is a glucose meter?


  ================================ RAG ================================ 
 - Latency:   0.06s
 - Chunks:  
   ['It determines the amount of glucose in the blood.']
 - Similarities:  
   [0.9550966024398804]
 - Metadata:  
   [{'answer': 'It determines the amount of glucose in the blood.',
     'chunked_file_id': '95',
     'complete_chunks': 'What does a glucose meter do?',
     'question': 'What does a glucose meter do?',
     'reranking_embedding': 'tensor(shape=torch.Size([1, 384]), dtype=torch.float32)',
     'source': 'Medical_HiRAG_chunks.json'}]

  ================================ LLM ================================ 
 - Prompt:   <|prompt|>You are an assistant, short answer using the following information: It determines the amount of glucose in the blood.
What is a glucose meter?</s><|answer|>
 - Answer:   A glucose meter is a device used to measure the amount of glucose (sugar) in the blood. It typically consists of a small, disposable test strip that is inserted into a blood g

Ask a question (or type 'q' to quit):  q


Exiting the program.
